Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIGURATION ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGES_PER_SESSION = 3
IMAGE_SIZE = (100, 300)
FUSION_METHOD = 'vstack'  # Options: 'vstack', 'hstack', '2x2'

fused_images = []
fused_labels = []

# ==== FUNCTION ====
def process_session_strategy1(base_path, session_label):
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Session {session_label}"):
        for img_idx in range(1, IMAGES_PER_SESSION + 1):
            images_2d = []
            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                print(f"📸 Loading: {img_path}")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Failed to load: {img_path}")
                    continue
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                images_2d.append(img_norm)

            if len(images_2d) == 4:
                if FUSION_METHOD == 'vstack':
                    fused_img = np.vstack(images_2d)
                elif FUSION_METHOD == 'hstack':
                    fused_img = np.hstack(images_2d)
                elif FUSION_METHOD == '2x2':
                    top = np.hstack([images_2d[0], images_2d[1]])
                    bottom = np.hstack([images_2d[2], images_2d[3]])
                    fused_img = np.vstack([top, bottom])
                else:
                    raise ValueError("Unsupported fusion method.")
                fused_images.append(fused_img)
                fused_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")

# ==== RUN FOR BOTH SESSIONS ====
process_session_strategy1(base_path_sess1, session_label=1)
process_session_strategy1(base_path_sess2, session_label=2)

# ==== OPTIONAL: Convert labels to NumPy array ====
fused_labels = np.array(fused_labels)
print(f"\n✅ Loaded {len(fused_images)} fused images total.")

# ==== COMPUTE (2D)²PCA PROJECTION MATRICES ====
def compute_2d2pca_projection(images, num_rows, num_cols):
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n

    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))

    for img in images:
        A = img - mean_img
        G_row += A @ A.T
        G_col += A.T @ A

    G_row /= n
    G_col /= n

    # Eigen decomposition
    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)

    # Sort and select top components
    idx_r = np.argsort(-eig_vals_r)
    idx_c = np.argsort(-eig_vals_c)
    U = eig_vecs_r[:, idx_r[:num_rows]]  # row projection
    V = eig_vecs_c[:, idx_c[:num_cols]]  # column projection

    return U, V

# ==== PROJECT IMAGES USING (2D)²PCA ====
num_row_components = 137
num_col_components = 137
U, V = compute_2d2pca_projection(fused_images, num_row_components, num_col_components)

projected_features = []
for img in fused_images:
    feat = U.T @ img @ V
    projected_features.append(feat)
    if len(projected_features) <= 3:
        print(f"🧮 Sample shape after (2D)²PCA: {feat.shape}")

# ====  FLATTEN FOR CLASSIFIER ====
flat_features = np.array([f.flatten() for f in projected_features])
print(f"\n✅ Final feature matrix shape: {flat_features.shape}")


Testing:

In [ ]:
test_data = []
test_labels = []
test_paths = []

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Preparing test data"):
    for img_idx in [4, 5, 6]:  # P1 protocol test images
        for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
            images_2d = []
            current_paths = []

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder_name, f"{img_idx:02d}.jpg")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Missing: {img_path}")
                    continue

                print(f"✅ Using: {img_path}")
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                images_2d.append(img_norm)
                current_paths.append(img_path)

            if len(images_2d) == 4:
                fused_img = np.vstack(images_2d)
                test_data.append(fused_img)
                test_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")
                test_paths.append(current_paths)
            else:
                print(f"⚠️ Incomplete: Subject {subject_id}, Image {img_idx}, Session {session_label}")

# Convert to array
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# ==== STEP X: PROJECT TEST IMAGES USING (2D)²PCA ====
proj_test_features = [U.T @ img @ V for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")


Benchmarking :

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img5_s2"

    # Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # Get the nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img2_s2"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Compare subject ID and session
    pred_id, pred_session = predicted_label.split('_')[0], predicted_label.split('_')[-1]
    true_id, true_session = true_label.split('_')[0], true_label.split('_')[-1]

    if pred_id == true_id and pred_session == true_session:
        correct_matches += 1
        print("  🟢 Match (ID & Session correct)")
    else:
        print("  🔴 Mismatch")

# Compute accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy: {accuracy:.2f}%")

Benchmarking 2:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance (Match: Subject ID only)...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img5_s2"

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # Find nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img2_s2"

    # Extract subject ID only (before "_img")
    pred_id = predicted_label.split('_')[0]  # e.g., "005"
    true_id = true_label.split('_')[0]       # e.g., "005"

    if pred_id == true_id:
        correct_matches += 1
        match_result = "✅ Match (Subject ID correct, session ignored)"
    else:
        match_result = "❌ Mismatch"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")
    print(f"  ➡️  Result   → {match_result}")

# Compute and print final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n📊 Final Results")
print(f"✅ Correct subject matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Subject only): {accuracy:.2f}%")
